In [3]:
# =====================================================================
# RESEARCH PROJECT
# Valuation of Corporate Financial Claims Under Alternative
# Capital Structures: An Extended Application of the Merton Model
#
# Interactive Google Colab Version
# Run the cell -> Fill the form -> Submit & Run Model
# =====================================================================

!pip -q install yfinance scipy ipywidgets

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import ipywidgets as widgets

from scipy.optimize import least_squares
from scipy.stats import norm
from IPython.display import display, clear_output, Markdown


# =====================================================================
# PART 1: MODEL SETTINGS
# =====================================================================

TRADING_DAYS = 252

MIN_EQUITY_VOL = 0.001
MIN_ASSET_VOL = 0.005
MAX_ASSET_VOL = 5.0

MAX_CALIBRATION_ERROR_PERCENT = 1.0


# =====================================================================
# PART 2: CREATE INPUT FORM
# =====================================================================

print("=" * 70)
print("MERTON MODEL - COMPANY INPUT FORM")
print("=" * 70)

print(
    "Enter the company information and annual-report figures below. "
    "All financial amounts must be in INR crore."
)

# Company information

company_name_input = widgets.Text(
    value="Vedanta Limited",
    description="Company:",
    placeholder="Enter company name",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

ticker_input = widgets.Text(
    value="VEDL.NS",
    description="Yahoo ticker:",
    placeholder="Example: VEDL.NS",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

# Dates

valuation_date_input = widgets.Text(
    value="2026-03-31",
    description="Valuation date:",
    placeholder="YYYY-MM-DD",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

volatility_start_input = widgets.Text(
    value="2021-04-01",
    description="Volatility start:",
    placeholder="YYYY-MM-DD",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

volatility_end_input = widgets.Text(
    value="2026-03-31",
    description="Volatility end:",
    placeholder="YYYY-MM-DD",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

# Annual report inputs

shares_input = widgets.Text(
    value="391.0388057",
    description="Shares (crore):",
    placeholder="Enter shares outstanding",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

senior_debt_input = widgets.Text(
    value="10648",
    description="Senior debt (₹ Cr):",
    placeholder="Enter senior debt",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

sub_debt_input = widgets.Text(
    value="0",
    description="Subordinated debt:",
    placeholder="Enter subordinated debt",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

maturity_input = widgets.Text(
    value="3",
    description="Maturity (years):",
    placeholder="Enter representative maturity",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

# Risk-free rate

gsec_input = widgets.Text(
    value="6.85",
    description="G-Sec yield (%):",
    placeholder="Enter historical government bond yield",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="500px")
)

submit_button = widgets.Button(
    description="Submit & Run Model",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="220px", height="40px")
)

form_output = widgets.Output()

input_form = widgets.VBox([
    widgets.HTML("<h3>1. Company Information</h3>"),
    company_name_input,
    ticker_input,

    widgets.HTML("<h3>2. Valuation and Volatility Dates</h3>"),
    valuation_date_input,
    volatility_start_input,
    volatility_end_input,

    widgets.HTML("<h3>3. Annual Report Information</h3>"),
    shares_input,
    senior_debt_input,
    sub_debt_input,
    maturity_input,

    widgets.HTML("<h3>4. Risk-Free Rate</h3>"),
    gsec_input,

    widgets.HTML(
        "<p><b>Note:</b> Use INR crore for financial amounts and "
        "crore shares for shares outstanding. The government bond "
        "yield should correspond to the valuation date.</p>"
    ),

    submit_button,
    form_output
])

display(input_form)


# =====================================================================
# PART 3: MODEL FUNCTIONS
# =====================================================================

def calculate_d1_d2(V, K, r, sigma, T):

    if V <= 0 or sigma <= 0 or T <= 0:
        raise ValueError("Invalid Black-Scholes input.")

    if K <= 0:
        return np.inf, np.inf

    d1 = (
        np.log(V / K) + (r + 0.5 * sigma**2) * T
    ) / (sigma * np.sqrt(T))

    d2 = d1 - sigma * np.sqrt(T)

    return d1, d2


def black_scholes_call(V, K, r, sigma, T):

    if K <= 0:
        return float(V)

    d1, d2 = calculate_d1_d2(V, K, r, sigma, T)

    value = (
        V * norm.cdf(d1)
        - K * np.exp(-r * T) * norm.cdf(d2)
    )

    return float(max(value, 0))


def black_scholes_put(V, K, r, sigma, T):

    if K <= 0:
        return 0.0

    d1, d2 = calculate_d1_d2(V, K, r, sigma, T)

    value = (
        K * np.exp(-r * T) * norm.cdf(-d2)
        - V * norm.cdf(-d1)
    )

    return float(max(value, 0))


def merton_equations(
    parameters,
    debt,
    observed_equity,
    observed_equity_volatility,
    risk_free_rate,
    maturity_years
):

    log_V, log_sigma_V = parameters

    V = np.exp(log_V)
    sigma_V = np.exp(log_sigma_V)

    try:
        d1, d2 = calculate_d1_d2(
            V,
            debt,
            risk_free_rate,
            sigma_V,
            maturity_years
        )

        model_equity = (
            V * norm.cdf(d1)
            - debt * np.exp(-risk_free_rate * maturity_years)
            * norm.cdf(d2)
        )

        if model_equity <= 0 or not np.isfinite(model_equity):
            return np.array([1e6, 1e6])

        model_equity_volatility = (
            V * norm.cdf(d1) * sigma_V / model_equity
        )

        residual_equity = (
            model_equity - observed_equity
        ) / observed_equity

        residual_volatility = (
            model_equity_volatility - observed_equity_volatility
        ) / observed_equity_volatility

        result = np.array([
            residual_equity,
            residual_volatility
        ])

        if not np.all(np.isfinite(result)):
            return np.array([1e6, 1e6])

        return result

    except Exception:
        return np.array([1e6, 1e6])


def calculate_claims(V, senior, subordinated, r, sigma, T):

    total_debt = senior + subordinated

    # Senior debt: risk-free senior claim minus put option
    senior_value = (
        senior * np.exp(-r * T)
        - black_scholes_put(V, senior, r, sigma, T)
    )

    # Subordinated debt: call spread
    subordinated_value = (
        black_scholes_call(V, senior, r, sigma, T)
        - black_scholes_call(V, total_debt, r, sigma, T)
    )

    # Equity: residual call option
    equity_value = black_scholes_call(
        V, total_debt, r, sigma, T
    )

    total_claims = (
        senior_value + subordinated_value + equity_value
    )

    return {
        "Senior Debt": senior_value,
        "Subordinated Debt": subordinated_value,
        "Equity": equity_value,
        "Total Claims": total_claims,
        "Asset Value": V,
        "Reconciliation Error": total_claims - V
    }


# =====================================================================
# PART 4: MAIN MODEL EXECUTION FUNCTION
# =====================================================================

def run_merton_model(inputs):

    company_name = inputs["company"]
    ticker = inputs["ticker"]

    valuation_date = inputs["valuation_date"]
    start_date = inputs["start_date"]
    end_date = inputs["end_date"]

    shares_crore = inputs["shares"]
    senior_debt = inputs["senior_debt"]
    subordinated_debt = inputs["sub_debt"]
    maturity_years = inputs["maturity"]

    annual_yield = inputs["gsec"] / 100

    total_debt = senior_debt + subordinated_debt

    # Effective annual yield to continuous compounding
    risk_free_rate = np.log1p(annual_yield)

    if annual_yield <= -1:
        raise ValueError("Government bond yield must be above -100%.")

    if total_debt <= 0:
        raise ValueError("Total debt must be greater than zero.")

    if shares_crore <= 0:
        raise ValueError("Shares outstanding must be positive.")

    if maturity_years <= 0:
        raise ValueError("Debt maturity must be positive.")

    if senior_debt < 0 or subordinated_debt < 0:
        raise ValueError("Debt amounts cannot be negative.")

    if start_date >= end_date:
        raise ValueError(
            "Volatility start date must be before end date."
        )

    if end_date > valuation_date:
        raise ValueError(
            "Volatility end date cannot be after valuation date."
        )

    print("\n" + "=" * 70)
    print("RUNNING MERTON MODEL")
    print("=" * 70)

    # -------------------------------------------------------------
    # A. Download historical prices
    # -------------------------------------------------------------

    print("\nDownloading historical market data...")

    # Robust ticker formatting fix (e.g. converting 'NS.VEDL' to 'VEDL.NS')
    if ticker.startswith("NS."):
        ticker = ticker[3:] + ".NS"

    stock = yf.Ticker(ticker)
    price_data = stock.history(period="max", auto_adjust=False)

    if price_data.empty:
        raise ValueError(
            f"Yahoo Finance returned no historical data for '{ticker}'. "
            "Please verify that the ticker is valid on Yahoo Finance."
        )

    price_data = price_data.sort_index()

    if price_data.index.tz is not None:
        price_data.index = price_data.index.tz_localize(None)

    price_data = price_data.dropna(subset=["Close"])

    # -------------------------------------------------------------
    # B. Find valuation-date share price
    # -------------------------------------------------------------

    valuation_prices = price_data[
        price_data.index.normalize() <= valuation_date
    ]

    if valuation_prices.empty:
        raise ValueError(
            "No historical share price found on or before "
            "the valuation date."
        )

    valuation_price = float(
        valuation_prices["Close"].iloc[-1]
    )

    valuation_price_date = valuation_prices.index[-1].date()

    if valuation_price <= 0 or not np.isfinite(valuation_price):
        raise ValueError("Invalid historical share price.")

    market_equity = valuation_price * shares_crore

    # -------------------------------------------------------------
    # C. Estimate equity volatility
    # -------------------------------------------------------------

    volatility_data = price_data[
        (price_data.index.normalize() >= start_date)
        &
        (price_data.index.normalize() <= end_date)
    ].copy()

    if len(volatility_data) < 30:
        raise ValueError(
            "Insufficient historical price observations. "
            "Select a longer period."
        )

    price_series = volatility_data["Adj Close"].dropna()

    daily_returns = np.log(
        price_series / price_series.shift(1)
    ).dropna()

    if len(daily_returns) < 20:
        raise ValueError("Insufficient daily return observations.")

    equity_volatility = float(
        daily_returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    )

    if (
        not np.isfinite(equity_volatility)
        or equity_volatility <= MIN_EQUITY_VOL
    ):
        raise ValueError(
            "Equity volatility is zero or too close to zero. "
            "Review the ticker and historical period."
        )

    # -------------------------------------------------------------
    # D. Display market data
    # -------------------------------------------------------------

    print("\nMARKET DATA")

    print(f"Company: {company_name}")
    print(f"Ticker: {ticker}")
    print(f"Valuation date: {valuation_date.date()}")
    print(f"Share price date: {valuation_price_date}")
    print(f"Share price: ₹{valuation_price:,.2f}")
    print(f"Shares outstanding: {shares_crore:,.4f} crore")
    print(f"Market equity: ₹{market_equity:,.2f} crore")

    print(f"\nVolatility start: {start_date.date()}")
    print(f"Volatility end: {end_date.date()}")
    print(f"Price observations: {len(price_series)}")
    print(f"Daily return observations: {len(daily_returns)}")
    print(f"Equity volatility: {equity_volatility:.4%}")

    print("\nANNUAL REPORT DATA")

    print(f"Senior debt: ₹{senior_debt:,.2f} crore")
    print(f"Subordinated debt: ₹{subordinated_debt:,.2f} crore")
    print(f"Total debt: ₹{total_debt:,.2f} crore")
    print(f"Representative maturity: {maturity_years:.2f} years")
    print(f"Government bond yield: {annual_yield:.4%}")
    print(f"Continuous risk-free rate: {risk_free_rate:.4%}")

    # -------------------------------------------------------------
    # E. Merton calibration
    # -------------------------------------------------------------

    print("\nCalibrating asset value and asset volatility...")

    initial_asset_value = market_equity + total_debt

    initial_asset_volatility = max(
        MIN_ASSET_VOL * 2,
        equity_volatility * market_equity / initial_asset_value
    )

    initial_asset_volatility = min(
        initial_asset_volatility,
        1.0
    )

    lower_asset_bound = max(
        initial_asset_value * 1e-8,
        1e-8
    )

    upper_asset_bound = max(
        100 * (market_equity + total_debt),
        initial_asset_value * 10,
        1.0
    )

    lower_bounds = np.log([
        lower_asset_bound,
        MIN_ASSET_VOL
    ])

    upper_bounds = np.log([
        upper_asset_bound,
        MAX_ASSET_VOL
    ])

    initial_parameters = np.log([
        initial_asset_value,
        initial_asset_volatility
    ])

    calibration = least_squares(
        merton_equations,
        initial_parameters,
        bounds=(lower_bounds, upper_bounds),
        args=(
            total_debt,
            market_equity,
            equity_volatility,
            risk_free_rate,
            maturity_years
        ),
        max_nfev=30000,
        ftol=1e-12,
        xtol=1e-12,
        gtol=1e-12
    )

    if not calibration.success:
        raise RuntimeError(
            f"Calibration failed: {calibration.message}"
        )

    asset_value = float(np.exp(calibration.x[0]))
    asset_volatility = float(np.exp(calibration.x[1]))

    if (
        not np.isfinite(asset_value)
        or not np.isfinite(asset_volatility)
        or asset_value <= 0
        or asset_volatility <= 0
    ):
        raise RuntimeError("Invalid calibrated asset values.")

    if (
        asset_volatility <= MIN_ASSET_VOL * 1.001
        or asset_volatility >= MAX_ASSET_VOL * 0.999
    ):
        raise RuntimeError(
            "Calibrated asset volatility is too close to a "
            "numerical boundary. Review the inputs."
        )

    d1_cal, d2_cal = calculate_d1_d2(
        asset_value,
        total_debt,
        risk_free_rate,
        asset_volatility,
        maturity_years
    )

    fitted_equity_value = black_scholes_call(
        asset_value,
        total_debt,
        risk_free_rate,
        asset_volatility,
        maturity_years
    )

    fitted_equity_volatility = (
        asset_value
        * norm.cdf(d1_cal)
        * asset_volatility
        / fitted_equity_value
    )

    equity_fit_error = (
        (fitted_equity_value - market_equity)
        / market_equity
    ) * 100

    volatility_fit_error = (
        (fitted_equity_volatility - equity_volatility)
        / equity_volatility
    ) * 100

    if (
        not np.isfinite(equity_fit_error)
        or not np.isfinite(volatility_fit_error)
    ):
        raise RuntimeError("Invalid calibration diagnostics.")

    if (
        abs(equity_fit_error) > MAX_CALIBRATION_ERROR_PERCENT
        or abs(volatility_fit_error) > MAX_CALIBRATION_ERROR_PERCENT
    ):
        raise RuntimeError(
            "Calibration error exceeds the 1% tolerance. "
            f"Equity error: {equity_fit_error:.4f}%; "
            f"volatility error: {volatility_fit_error:.4f}%."
        )

    print("\nCALIBRATION RESULTS")

    print(f"Estimated asset value: ₹{asset_value:,.2f} crore")
    print(f"Estimated asset volatility: {asset_volatility:.4%}")
    print(f"Observed equity value: ₹{market_equity:,.2f} crore")
    print(f"Model equity value: ₹{fitted_equity_value:,.2f} crore")
    print(f"Observed equity volatility: {equity_volatility:.4%}")
    print(f"Model equity volatility: {fitted_equity_volatility:.4%}")
    print(f"Equity fit error: {equity_fit_error:.6f}%")
    print(f"Volatility fit error: {volatility_fit_error:.6f}%")

    # -------------------------------------------------------------
    # F. Financial claim valuation
    # -------------------------------------------------------------

    base_claims = calculate_claims(
        asset_value,
        senior_debt,
        subordinated_debt,
        risk_free_rate,
        asset_volatility,
        maturity_years
    )

    claims_table = pd.DataFrame({
        "Financial Claim": [
            "Senior Debt",
            "Subordinated Debt",
            "Equity",
            "Total Claims",
            "Estimated Asset Value"
        ],
        "Value (₹ crore)": [
            base_claims["Senior Debt"],
            base_claims["Subordinated Debt"],
            base_claims["Equity"],
            base_claims["Total Claims"],
            base_claims["Asset Value"]
        ]
    })

    print("\nVALUATION OF FINANCIAL CLAIMS")
    display(claims_table.round(4))

    print(
        "Reconciliation error: "
        f"₹{base_claims['Reconciliation Error']:.10f} crore"
    )

    # -------------------------------------------------------------
    # G. Default risk
    # -------------------------------------------------------------

    risk_neutral_pd = norm.cdf(-d2_cal)
    distance_to_default = d2_cal

    print("\nDEFAULT RISK INDICATORS")
    print(f"Distance to default (d2): {distance_to_default:.4f}")
    print(f"Risk-neutral PD: {risk_neutral_pd:.4%}")

    # -------------------------------------------------------------
    # H. Alternative capital structure analysis
    # -------------------------------------------------------------

    scenario_multipliers = [0.75, 1.00, 1.25, 1.50]

    scenario_results = []

    for multiplier in scenario_multipliers:

        test_senior = senior_debt * multiplier
        test_sub = subordinated_debt * multiplier
        test_total = test_senior + test_sub

        result = calculate_claims(
            asset_value,
            test_senior,
            test_sub,
            risk_free_rate,
            asset_volatility,
            maturity_years
        )

        _, test_d2 = calculate_d1_d2(
            asset_value,
            test_total,
            risk_free_rate,
            asset_volatility,
            maturity_years
        )

        scenario_results.append({
            "Debt Scenario": f"{multiplier:.0%}",
            "Total Debt": test_total,
            "Senior Debt Value": result["Senior Debt"],
            "Subordinated Debt Value": result["Subordinated Debt"],
            "Equity Value": result["Equity"],
            "Risk-neutral PD": norm.cdf(-test_d2)
        })

    scenario_df = pd.DataFrame(scenario_results)

    print("\nALTERNATIVE CAPITAL STRUCTURE ANALYSIS")
    display(scenario_df.round(4))

    # -------------------------------------------------------------
    # I. Volatility sensitivity
    # -------------------------------------------------------------

    volatility_multipliers = [0.75, 0.90, 1.00, 1.10, 1.25]

    volatility_results = []

    for multiplier in volatility_multipliers:

        test_sigma = asset_volatility * multiplier

        result = calculate_claims(
            asset_value,
            senior_debt,
            subordinated_debt,
            risk_free_rate,
            test_sigma,
            maturity_years
        )

        _, test_d2 = calculate_d1_d2(
            asset_value,
            total_debt,
            risk_free_rate,
            test_sigma,
            maturity_years
        )

        volatility_results.append({
            "Asset Volatility": test_sigma,
            "Senior Debt Value": result["Senior Debt"],
            "Subordinated Debt Value": result["Subordinated Debt"],
            "Equity Value": result["Equity"],
            "Risk-neutral PD": norm.cdf(-test_d2)
        })

    volatility_df = pd.DataFrame(volatility_results)

    print("\nVOLATILITY SENSITIVITY ANALYSIS")
    display(volatility_df.round(4))

    # -------------------------------------------------------------
    # J. Maturity sensitivity
    # -------------------------------------------------------------

    maturity_multipliers = [0.50, 0.75, 1.00, 1.25, 1.50]

    maturity_results = []

    for multiplier in maturity_multipliers:

        test_T = maturity_years * multiplier

        result = calculate_claims(
            asset_value,
            senior_debt,
            subordinated_debt,
            risk_free_rate,
            asset_volatility,
            test_T
        )

        _, test_d2 = calculate_d1_d2(
            asset_value,
            total_debt,
            risk_free_rate,
            asset_volatility,
            test_T
        )

        maturity_results.append({
            "Maturity (Years)": test_T,
            "Senior Debt Value": result["Senior Debt"],
            "Subordinated Debt Value": result["Subordinated Debt"],
            "Equity Value": result["Equity"],
            "Risk-neutral PD": norm.cdf(-test_d2)
        })

    maturity_df = pd.DataFrame(maturity_results)

    print("\nMATURITY SENSITIVITY ANALYSIS")
    display(maturity_df.round(4))

    # -------------------------------------------------------------
    # K. Leverage sensitivity
    # -------------------------------------------------------------

    leverage_multipliers = np.linspace(0.50, 2.00, 10)

    leverage_results = []

    for multiplier in leverage_multipliers:

        test_senior = senior_debt * multiplier
        test_sub = subordinated_debt * multiplier
        test_total = test_senior + test_sub

        result = calculate_claims(
            asset_value,
            test_senior,
            test_sub,
            risk_free_rate,
            asset_volatility,
            maturity_years
        )

        _, test_d2 = calculate_d1_d2(
            asset_value,
            test_total,
            risk_free_rate,
            asset_volatility,
            maturity_years
        )

        leverage_results.append({
            "Debt Multiplier": multiplier,
            "Total Debt": test_total,
            "Senior Debt Value": result["Senior Debt"],
            "Subordinated Debt Value": result["Subordinated Debt"],
            "Equity Value": result["Equity"],
            "Risk-neutral PD": norm.cdf(-test_d2)
        })

    leverage_df = pd.DataFrame(leverage_results)

    print("\nLEVERAGE SENSITIVITY ANALYSIS")
    display(leverage_df.round(4))

    # -------------------------------------------------------------
    # L. Charts
    # -------------------------------------------------------------

    plt.figure(figsize=(9, 5))

    plt.bar(
        ["Senior Debt", "Subordinated Debt", "Equity"],
        [
            base_claims["Senior Debt"],
            base_claims["Subordinated Debt"],
            base_claims["Equity"]
        ]
    )

    plt.title("Valuation of Financial Claims")
    plt.ylabel("Value (₹ crore)")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    for column in [
        "Senior Debt Value",
        "Subordinated Debt Value",
        "Equity Value"
    ]:
        plt.plot(
            scenario_df["Debt Scenario"],
            scenario_df[column],
            marker="o",
            label=column
        )

    plt.title("Impact of Alternative Capital Structures")
    plt.xlabel("Debt Scenario")
    plt.ylabel("Claim Value (₹ crore)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    for column in [
        "Senior Debt Value",
        "Subordinated Debt Value",
        "Equity Value"
    ]:
        plt.plot(
            volatility_df["Asset Volatility"] * 100,
            volatility_df[column],
            marker="o",
            label=column
        )

    plt.title("Impact of Asset Volatility on Claim Values")
    plt.xlabel("Asset Volatility (%)")
    plt.ylabel("Claim Value (₹ crore)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    for column in [
        "Senior Debt Value",
        "Subordinated Debt Value",
        "Equity Value"
    ]:
        plt.plot(
            maturity_df["Maturity (Years)"],
            maturity_df[column],
            marker="o",
            label=column
        )

    plt.title("Impact of Maturity on Claim Values")
    plt.xlabel("Maturity (Years)")
    plt.ylabel("Claim Value (₹ crore)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    plt.plot(
        leverage_df["Total Debt"],
        leverage_df["Equity Value"],
        marker="o",
        label="Equity Value"
    )

    plt.plot(
        leverage_df["Total Debt"],
        leverage_df["Senior Debt Value"],
        marker="o",
        label="Senior Debt Value"
    )

    plt.title("Impact of Leverage on Claim Values")
    plt.xlabel("Total Debt (₹ crore)")
    plt.ylabel("Claim Value (₹ crore)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # M. Interactive simulator
    # -------------------------------------------------------------

    print("\nINTERACTIVE CAPITAL STRUCTURE SIMULATOR")

    debt_slider = widgets.FloatSlider(
        value=1.0,
        min=0.5,
        max=2.0,
        step=0.05,
        description="Debt multiplier:",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="75%")
    )

    volatility_slider = widgets.FloatSlider(
        value=asset_volatility * 100,
        min=max(MIN_ASSET_VOL * 100, asset_volatility * 50),
        max=min(MAX_ASSET_VOL * 100, max(asset_volatility * 200, 100)),
        step=1,
        description="Asset volatility (%):",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="75%")
    )

    maturity_slider = widgets.FloatSlider(
        value=maturity_years,
        min=max(0.1, maturity_years * 0.25),
        max=max(maturity_years * 2, maturity_years + 1),
        step=0.1,
        description="Maturity (years):",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="75%")
    )

    simulator_output = widgets.Output()

    def update_simulator(change=None):

        with simulator_output:
            clear_output(wait=True)

            multiplier = debt_slider.value
            test_sigma = volatility_slider.value / 100
            test_T = maturity_slider.value

            test_senior = senior_debt * multiplier
            test_sub = subordinated_debt * multiplier
            test_total = test_senior + test_sub

            result = calculate_claims(
                asset_value,
                test_senior,
                test_sub,
                risk_free_rate,
                test_sigma,
                test_T
            )

            _, test_d2 = calculate_d1_d2(
                asset_value,
                test_total,
                risk_free_rate,
                test_sigma,
                test_T
            )

            print("SIMULATED FINANCIAL CLAIM VALUES")
            print("-" * 45)
            print(f"Debt multiplier: {multiplier:.2f}x")
            print(f"Asset volatility: {test_sigma:.2%}")
            print(f"Maturity: {test_T:.2f} years")
            print()
            print(f"Senior debt: ₹{result['Senior Debt']:,.2f} crore")
            print(
                f"Subordinated debt: "
                f"₹{result['Subordinated Debt']:,.2f} crore"
            )
            print(f"Equity: ₹{result['Equity']:,.2f} crore")
            print(f"Total claims: ₹{result['Total Claims']:,.2f} crore")
            print(f"Asset value: ₹{asset_value:,.2f} crore")
            print(f"Risk-neutral PD: {norm.cdf(-test_d2):.4%}")
            print(
                f"Reconciliation error: "
                f"₹{result['Reconciliation Error']:.8f} crore"
            )

            # Dynamically display the simulated claim distribution chart
            fig, ax = plt.subplots(figsize=(8, 4.5))
            ax.bar(
                ["Senior Debt", "Subordinated Debt", "Equity"],
                [result["Senior Debt"], result["Subordinated Debt"], result["Equity"]],
                color=["#1f77b4", "#ff7f0e", "#2ca02c"]
            )
            ax.set_title(f"Simulated Claims Distribution (Multiplier: {multiplier:.2f}x)")
            ax.set_ylabel("Value (₹ crore)")
            ax.grid(axis="y", alpha=0.3)
            plt.tight_layout()
            plt.show()

    debt_slider.observe(update_simulator, names="value")
    volatility_slider.observe(update_simulator, names="value")
    maturity_slider.observe(update_simulator, names="value")

    display(debt_slider)
    display(volatility_slider)
    display(maturity_slider)
    display(simulator_output)

    update_simulator()

    # -------------------------------------------------------------
    # N. Automatic findings
    # -------------------------------------------------------------

    print("\n" + "=" * 70)
    print("RESEARCH FINDINGS")
    print("=" * 70)

    print(f"Company: {company_name}")
    print(f"Valuation date: {valuation_date.date()}")

    print(f"Estimated asset value: ₹{asset_value:,.2f} crore")
    print(f"Estimated asset volatility: {asset_volatility:.2%}")

    print(
        f"Senior debt value: ₹{base_claims['Senior Debt']:,.2f} crore"
    )

    print(
        f"Subordinated debt value: "
        f"₹{base_claims['Subordinated Debt']:,.2f} crore"
    )

    print(f"Equity value: ₹{base_claims['Equity']:,.2f} crore")
    print(f"Risk-neutral PD: {risk_neutral_pd:.2%}")

    base_equity = float(
        scenario_df.loc[
            scenario_df["Debt Scenario"] == "100%",
            "Equity Value"
        ].iloc[0]
    )

    higher_debt_equity = float(
        scenario_df.loc[
            scenario_df["Debt Scenario"] == "125%",
            "Equity Value"
        ].iloc[0]
    )

    if higher_debt_equity < base_equity:
        print(
            "A 25% increase in debt reduces modelled equity value "
            "under the selected assumptions."
        )
    elif higher_debt_equity > base_equity:
        print(
            "A 25% increase in debt increases modelled equity value "
            "under the selected assumptions."
        )

    # -------------------------------------------------------------
    # O. Model limitations
    # -------------------------------------------------------------

    print("\nMODEL LIMITATIONS")

    limitations = [
        "Total debt is treated as one unified default threshold during "
        "standard Merton calibration.",

        "Senior and subordinated claims are separated after calibration "
        "using a common-maturity priority waterfall.",

        "The model assumes a lognormal asset-value process and constant "
        "asset volatility and risk-free rate.",

        "Risk-neutral PD is not a real-world default forecast.",

        "Alternative capital structure scenarios hold asset value and "
        "asset volatility constant rather than recalibrating them.",

        "The representative maturity simplifies the company's actual "
        "debt maturity schedule.",

        "Debt priority must be supported by contractual information. "
        "Unsecured debt is not automatically subordinated."
    ]

    for number, limitation in enumerate(limitations, 1):
        print(f"{number}. {limitation}")

    # -------------------------------------------------------------
    # P. Final summary
    # -------------------------------------------------------------

    final_summary = pd.DataFrame({
        "Parameter": [
            "Company",
            "Ticker",
            "Valuation Date",
            "Volatility Start Date",
            "Volatility End Date",
            "Share Price Date",
            "Historical Share Price",
            "Shares Outstanding (crore)",
            "Market Equity (₹ crore)",
            "Senior Debt (₹ crore)",
            "Subordinated Debt (₹ crore)",
            "Total Debt (₹ crore)",
            "Maturity (years)",
            "Government Bond Yield",
            "Risk-free Rate (continuous)",
            "Equity Volatility",
            "Asset Value (₹ crore)",
            "Asset Volatility",
            "Senior Debt Value (₹ crore)",
            "Subordinated Debt Value (₹ crore)",
            "Equity Value (₹ crore)",
            "Risk-neutral PD"
        ],
        "Value": [
            company_name,
            ticker,
            str(valuation_date.date()),
            str(start_date.date()),
            str(end_date.date()),
            str(valuation_price_date),
            round(valuation_price, 4),
            round(shares_crore, 4),
            round(market_equity, 4),
            round(senior_debt, 4),
            round(subordinated_debt, 4),
            round(total_debt, 4),
            maturity_years,
            f"{annual_yield * 100:.4f}%",
            round(risk_free_rate, 6),
            round(equity_volatility, 6),
            round(asset_value, 4),
            round(asset_volatility, 6),
            round(base_claims["Senior Debt"], 4),
            round(base_claims["Subordinated Debt"], 4),
            round(base_claims["Equity"], 4),
            round(risk_neutral_pd, 6)
        ]
    })

    print("\nFINAL RESEARCH MODEL SUMMARY")
    display(final_summary)

    print("\nMODEL EXECUTION COMPLETED SUCCESSFULLY.")


# =====================================================================
# PART 5: FORM SUBMISSION AND VALIDATION
# =====================================================================

def submit_form(button):

    with form_output:
        clear_output(wait=True)

        try:
            # Read all form values
            company = company_name_input.value.strip()
            ticker = ticker_input.value.strip().upper()

            if not company:
                raise ValueError("Enter a company name.")

            if not ticker:
                raise ValueError("Enter a Yahoo Finance ticker.")

            # Clean up inverted Indian tickers (e.g., NS.VEDL -> VEDL.NS)
            if ticker.startswith("NS."):
                ticker = ticker[3:] + ".NS"

            valuation_date = pd.Timestamp(
                valuation_date_input.value.strip()
            ).normalize()

            start_date = pd.Timestamp(
                volatility_start_input.value.strip()
            ).normalize()

            end_date = pd.Timestamp(
                volatility_end_input.value.strip()
            ).normalize()

            shares = float(shares_input.value)
            senior = float(senior_debt_input.value)
            subordinated = float(sub_debt_input.value)
            maturity = float(maturity_input.value)
            gsec = float(gsec_input.value)

            numeric_values = [
                shares, senior, subordinated, maturity, gsec
            ]

            if not np.all(np.isfinite(numeric_values)):
                raise ValueError("All numeric inputs must be finite.")

            if shares <= 0:
                raise ValueError("Shares outstanding must be positive.")

            if senior < 0 or subordinated < 0:
                raise ValueError("Debt values cannot be negative.")

            if senior + subordinated <= 0:
                raise ValueError("Total debt must be greater than zero.")

            if maturity <= 0:
                raise ValueError("Maturity must be positive.")

            if gsec <= -100:
                raise ValueError("G-Sec yield must be above -100%.")

            if start_date >= end_date:
                raise ValueError(
                    "Volatility start date must be before end date."
                )

            if end_date > valuation_date:
                raise ValueError(
                    "Volatility end date cannot be after valuation date."
                )

            inputs = {
                "company": company,
                "ticker": ticker,
                "valuation_date": valuation_date,
                "start_date": start_date,
                "end_date": end_date,
                "shares": shares,
                "senior_debt": senior,
                "sub_debt": subordinated,
                "maturity": maturity,
                "gsec": gsec
            }

            print("Inputs validated successfully.")
            print("Starting model calculations...")

            run_merton_model(inputs)

        except Exception as error:
            print("\nMODEL COULD NOT RUN")
            print(f"Reason: {error}")
            print(
                "\nCheck your dates, annual-report figures, "
                "government bond yield and Yahoo Finance ticker."
            )


submit_button.on_click(submit_form)


MERTON MODEL - COMPANY INPUT FORM
Enter the company information and annual-report figures below. All financial amounts must be in INR crore.
